# 🤖 Function Calling & Agentic Pattern — Multi-Model Benchmark

Evaluates small LLMs on function calling and agentic behaviour.

Uses each model's **own tool contract**:
- template references `tools` (Qwen, nesso-agentic) -> `apply_chat_template(tools=...)`
- template does NOT (zagreus compact / chatanchor models) -> the compact `<tools>\n[json]\n</tools>` block is **injected into the system message**, exactly as in their SFT data.
  (Passing `tools=` to those templates silently drops the tools - the model then never sees them.)

### Agentic turn loop
```
user query
  → chat(messages, tools=TOOLS)          # model decides: call a tool or answer directly
  → parse_tool_call(answer)              # extract {name, arguments}
  → run_tool(name, args)                 # mock execution → result dict
  → chat(messages2, tools=None)          # final natural-language answer
```

### Evaluation dimensions
| Dimension | What we check |
|---|---|
| **Tag compliance** | Response wraps call in `<tool_call>` tags |
| **Tool selection** | Correct tool chosen |
| **Argument correctness** | Required args present |
| **No-tool refusal** | No spurious call when none needed |
| **Multi-step chaining** | Second call after `<tool_response>` injection |

## 📦 Imports

In [1]:
import torch
import json
import re
import ast
import gc
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown

~/ai/ai2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ⚙️ Configuration

In [2]:
# ── Models to benchmark ──────────────────────────────────────
MODELS = [
    "giux78/zagreus-0.4B-cpt-full-32k-sft-compact-v5b",
    "giux78/zagreus-0.4B-cpt-full-32k-sft-compact-v6",
    "Qwen/Qwen3-0.6B",
]

DEFAULT_MAX_TOKENS = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device : cuda
GPU    : NVIDIA A100 80GB PCIe
VRAM   : 85.1 GB


## 🔧 Model Loading

Loads one model at a time. To inspect the full chat template call `load_model(model_id, show_template=True)`.

In [3]:
def load_model(model_id, show_template=False):
    """Load tokenizer + model. Pass show_template=True to print the chat template."""
    print(f"  loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    print(f"  loading weights...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        # bf16, not fp16: these models (zagreus / Qwen / nesso) are bf16-native;
        # fp16 can degrade them (overflow) — fairer to load each in its own dtype.
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    if not torch.cuda.is_available():
        model = model.to(DEVICE)
    model.eval()

    tmpl = tokenizer.chat_template
    print(f"  chat_template : {'present' if tmpl else 'MISSING'}"
          + (f" ({len(tmpl)} chars)" if tmpl else ""))
    if show_template and tmpl:
        print("\n--- chat_template ---")
        print(tmpl)
        print("--- end ---\n")
    return tokenizer, model


def unload_model(model):
    """Free GPU/CPU memory after each model."""
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


print("load_model / unload_model defined  (bf16-native load).")

load_model / unload_model defined  (bf16-native load).


## 💬 `chat()` — Prompt Building & Generation

Mirrors the reference notebook exactly:
- `tokenizer.apply_chat_template(tools=tools)` — native schema injection
- `do_sample=False` — deterministic, reproducible
- Decodes full text with special tokens, then **regex-extracts the last `<|im_start|>assistant` block**
- Prints `=== RAW OUTPUT ===` and `=== PARSED ASSISTANT ===` for every call (useful for debugging)

In [4]:
def has_native_tools(tokenizer):
    """True if the chat template itself renders `tools` (Qwen, nesso-agentic).
    Zagreus compact/chatanchor templates do not -> tools must be injected into the system message."""
    return bool(re.search(r"\btools\b", tokenizer.chat_template or ""))


def compact_tools_block(tools):
    """The compact serving contract used in the compact-* SFT data."""
    return "<tools>\n" + json.dumps(tools, separators=(",", ":"), ensure_ascii=False) + "\n</tools>"


def inject_tools(messages, tools):
    """Append the compact <tools> block to the system message (create one if absent)."""
    block = compact_tools_block(tools)
    msgs = [dict(m) for m in messages]
    if msgs and msgs[0]["role"] == "system":
        msgs[0]["content"] = msgs[0]["content"].rstrip() + "\n" + block
    else:
        msgs.insert(0, {"role": "system", "content": block})
    return msgs


def chat(tokenizer, model, messages, tools=None, max_tokens=DEFAULT_MAX_TOKENS, verbose=False):
    """
    Generate one turn using the model's OWN tool contract:
      - template renders tools  -> apply_chat_template(tools=tools)
      - template ignores tools  -> inject compact <tools> block in system (as in SFT data)
    Single BOS (tokenize inside apply_chat_template) and decode of new tokens only.
    """
    if tools and not has_native_tools(tokenizer):
        messages, tools = inject_tools(messages, tools), None

    enc = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,    # Qwen3: no <think> (non-thinking mode); ignored by other templates
    ).to(model.device)

    outputs = model.generate(
        **enc,
        max_new_tokens=max_tokens,
        do_sample=False,          # greedy / deterministic
        temperature=None,
        top_p=None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    gen    = outputs[0][enc["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen, skip_special_tokens=False)
    answer = re.split(r"<\|eot_id\|>|<\|im_end\|>|<\|end_of_text\|>", answer)[0]
    answer = re.sub(r"<think>.*?</think>\s*", "", answer, flags=re.S).strip()  # drop Qwen CoT

    if verbose:
        print("\n=== PROMPT ===")
        print(tokenizer.decode(enc["input_ids"][0], skip_special_tokens=False))
        print("\n=== PARSED ASSISTANT ===")
        print(answer)

    return answer


print("chat() defined  (native-contract detection: tools= vs compact system injection).")

chat() defined  (native-contract detection: tools= vs compact system injection).


## 🛠️ Tool Definitions & Mock Executor

Tools are defined in **OpenAI function-calling schema** and passed directly to  
`apply_chat_template(tools=TOOLS)` — the tokenizer injects them into `<tools>...</tools>` automatically.

`run_tool()` is the mock executor. It mirrors `run_tool()` from the reference notebook,  
with alias normalisation (e.g. `type` → `kind` for `convert_units`).

In [5]:
# ── Tool schema (OpenAI function-calling format) ──────────────
# NAMES aligned to common/training-consistent conventions:
#   calculate (was calculator), web_search (was search_web), create_event (was create_calendar_event)
TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'London'"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
        }, "required": ["city"]},
    }},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate a mathematical expression and return the numeric result.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Math expression, e.g. '(12+5)*3'"},
        }, "required": ["expression"]},
    }},
    {"type": "function", "function": {
        "name": "convert_units",
        "description": "Convert units. kind must be 'km_to_mi' or 'c_to_f'.",
        "parameters": {"type": "object", "properties": {
            "kind":  {"type": "string", "enum": ["km_to_mi", "c_to_f"]},
            "value": {"type": "number"},
        }, "required": ["kind", "value"]},
    }},
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the web for up-to-date information.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"},
        }, "required": ["query"]},
    }},
    {"type": "function", "function": {
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "parameters": {"type": "object", "properties": {
            "to":      {"type": "string"},
            "subject": {"type": "string"},
            "body":    {"type": "string"},
        }, "required": ["to", "subject", "body"]},
    }},
    {"type": "function", "function": {
        "name": "get_stock_price",
        "description": "Get the latest stock price for a ticker symbol.",
        "parameters": {"type": "object", "properties": {
            "ticker":   {"type": "string"},
            "currency": {"type": "string"},
        }, "required": ["ticker"]},
    }},
    {"type": "function", "function": {
        "name": "create_event",
        "description": "Create a new calendar event.",
        "parameters": {"type": "object", "properties": {
            "title":            {"type": "string"},
            "date":             {"type": "string", "description": "ISO date, e.g. '2025-06-15'"},
            "time":             {"type": "string"},
            "duration_minutes": {"type": "integer"},
        }, "required": ["title", "date"]},
    }},
]

TOOL_NAMES = {t["function"]["name"] for t in TOOLS}


def run_tool(name, args):
    """Execute a tool call against mock data. Returns a result dict."""
    args = dict(args or {})
    if name == "get_weather":
        return {"city": args.get("city"), "temp_c": 18, "condition": "partly cloudy"}
    if name == "calculate":
        try:
            return {"result": eval(args["expression"], {"__builtins__": {}, "__import__": None})}
        except Exception as e:
            return {"error": str(e)}
    if name == "convert_units":
        if "kind" not in args and "type" in args:
            args["kind"] = args.pop("type")
        v = float(args.get("value", 0)); kind = args.get("kind", "")
        if kind == "km_to_mi": return {"result": round(v * 0.621371, 4)}
        if kind == "c_to_f":   return {"result": round((v * 9/5) + 32, 2)}
        return {"error": f"Unknown kind: {kind}"}
    if name == "web_search":
        return {"results": [{"title": "F1 Race Result",
                             "snippet": "Max Verstappen won the 2025 Monaco Grand Prix."}]}
    if name == "send_email":
        return {"status": "sent", "message_id": "msg_001"}
    if name == "get_stock_price":
        return {"ticker": args.get("ticker"), "price": 213.55,
                "currency": args.get("currency", "USD")}
    if name == "create_event":
        return {"status": "created", "event_id": "evt_001"}
    return {"error": f"Unknown tool: {name}"}


print(f"Registered {len(TOOLS)} tools: {sorted(TOOL_NAMES)}")

Registered 7 tools: ['calculate', 'convert_units', 'create_event', 'get_stock_price', 'get_weather', 'send_email', 'web_search']


## 💬 System Prompt

Short and intentional — the template already injects the full tool schema and format instructions.  
The system prompt only sets the **decision policy**.

In [6]:
# Short, compact-contract-style system contexts (the tool schema is added by chat()).
SYSTEM_PROMPTS = {
    "en": "You are an operations assistant. Use the available tools when needed.",
    "it": "Sei un assistente operativo. Usa gli strumenti disponibili quando serve.",
}
SYSTEM_PROMPT = SYSTEM_PROMPTS["en"]   # back-compat

def system_for(lang):
    return SYSTEM_PROMPTS.get(lang, SYSTEM_PROMPTS["en"])

print(SYSTEM_PROMPTS)

{'en': 'You are an operations assistant. Use the available tools when needed.', 'it': 'Sei un assistente operativo. Usa gli strumenti disponibili quando serve.'}


## 🔄 Tool-Call Parser & Agentic Turn

### `parse_tool_call(s)`
The model's native format (from the template) is:
```
<tool_call>
{"name": "<function-name>", "arguments": <args-json-object>}
</tool_call>
```
Parser: regex → `json.loads` → `ast.literal_eval` fallback (handles single-quote dicts).  
Also scans for bare balanced-brace JSON blocks as a last resort.

### `tool_turn(tokenizer, model, user_text)`
Full agentic loop from the reference notebook:
1. Step 1 — `chat(messages, tools=TOOLS)` — model decides
2. Parse → `run_tool()` → result
3. Step 2 — `chat(messages2, tools=None)` — final answer (tools disabled)

In [7]:
TOOL_NAME_KEYS = ("name", "tool", "tool_name", "function", "action")


def _try_parse(raw):
    """json.loads, falling back to ast.literal_eval (single-quote dicts)."""
    raw = raw.strip()
    try:
        obj = json.loads(raw)
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        pass
    try:
        obj = ast.literal_eval(raw)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    return None


def _balanced_blocks(text):
    """Yield every top-level balanced {...} block in text (string-aware)."""
    depth, begin, in_str, esc = 0, None, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"' and depth > 0:
            in_str = True
        elif ch == "{":
            if depth == 0: begin = i
            depth += 1
        elif ch == "}" and depth > 0:
            depth -= 1
            if depth == 0 and begin is not None:
                yield text[begin:i+1]; begin = None


def _normalise(d):
    """Canonical {name, arguments}; tolerates key variants and string-encoded args."""
    fn = d.get("function")
    name = (d.get("name") or d.get("tool") or d.get("tool_name") or d.get("action")
            or (fn.get("name") if isinstance(fn, dict) else fn if isinstance(fn, str) else None))
    args = d.get("arguments")
    if args is None: args = d.get("parameters")
    if args is None: args = d.get("args")
    if args is None: args = d.get("input")
    if args is None and isinstance(fn, dict): args = fn.get("arguments")
    if isinstance(args, str):
        args = _try_parse(args) or {}
    if not isinstance(args, dict):
        args = {}
    return {"name": name, "arguments": args} if isinstance(name, str) else None


def parse_tool_calls(s):
    """
    ALL tool calls in a model response (parallel calls supported), in order.
      1. every <tool_call>...</tool_call> body (also an UNCLOSED trailing <tool_call>,
         e.g. generation stopped at <|eot_id|> before the closing tag)
      2. fallback for tag-less output: bare JSON blocks with a name-like key
    """
    calls = []
    # 1. tagged calls — split on the opening tag so nested braces / missing close tags are fine
    if "<tool_call>" in s:
        for part in s.split("<tool_call>")[1:]:
            part = part.split("</tool_call>")[0]
            for block in _balanced_blocks(part):
                p = _try_parse(block)
                n = _normalise(p) if p else None
                if n:
                    calls.append(n); break
        if calls:
            return calls
    # 2. tag-less fallback
    for block in _balanced_blocks(s):
        p = _try_parse(block)
        if p and any(k in p for k in TOOL_NAME_KEYS):
            n = _normalise(p)
            if n: calls.append(n)
    return calls


def parse_tool_call(s):
    """First tool call or None (back-compat)."""
    calls = parse_tool_calls(s)
    return calls[0] if calls else None


def tool_turn(tokenizer, model, user_text, lang="en", verbose=False):
    """
    Agentic turn on the compact contract:
      1) step 1: model decides (tools in context)
      2) run_tool() for EVERY parsed call
      3) step 2: tool observation(s) appended, tools STILL in context (as in SFT data),
         no extra user nudge -> tests real observation grounding.
    Returns (step1_answer, first_tool_call, final_answer).
    """
    messages = [
        {"role": "system", "content": system_for(lang)},
        {"role": "user",   "content": user_text},
    ]
    a1 = chat(tokenizer, model, messages, tools=TOOLS, verbose=verbose)
    calls = parse_tool_calls(a1)
    if not calls:
        return a1, None, a1

    messages2 = messages + [{"role": "assistant", "content": a1}]
    for tc in calls:
        out = run_tool(tc["name"], tc["arguments"])
        if verbose:
            print(f"  tool called : {tc['name']}({tc['arguments']})\n  tool result : {out}")
        messages2.append({"role": "tool", "content": json.dumps(out, ensure_ascii=False)})

    final = chat(tokenizer, model, messages2, tools=TOOLS, verbose=verbose)
    return a1, calls[0], final


print("parse_tool_calls / parse_tool_call / tool_turn defined.")

parse_tool_calls / parse_tool_call / tool_turn defined.


## 🧪 Test Scenarios

| Category | Description |
|---|---|
| `single_call` | One clearly expected tool call |
| `argument_precision` | All required + optional args must be present |
| `tool_selection` | Right tool chosen among several candidates |
| `no_tool` | Model must answer directly — no call needed |
| `multi_step` | Two sequential calls: step 1 → mock result → step 2 |

In [8]:
# Bilingual scenarios. Each has a "lang" field ("en"/"it"). Tool names match TOOLS.
TEST_SCENARIOS = {
    # ═══════════════ ENGLISH ═══════════════
    "weather_basic":       {"lang":"en","category":"single_call","user":"What is the weather like in London right now?","expected_tool":"get_weather","expected_args":["city"]},
    "calculator_basic":    {"lang":"en","category":"single_call","user":"What is (347 + 58) * 12?","expected_tool":"calculate","expected_args":["expression"]},
    "stock_price":         {"lang":"en","category":"single_call","user":"What is Apple's current stock price?","expected_tool":"get_stock_price","expected_args":["ticker"]},
    "send_email_full":     {"lang":"en","category":"argument_precision","user":"Send an email to alice@example.com with subject 'Meeting tomorrow' and body 'Hi Alice, reminder about our 10am meeting. Best, Bob.'","expected_tool":"send_email","expected_args":["to","subject","body"]},
    "calendar_full_args":  {"lang":"en","category":"argument_precision","user":"Create a 90-minute calendar event called 'Project Review' on 2025-07-10 at 14:30.","expected_tool":"create_event","expected_args":["title","date","time","duration_minutes"]},
    "weather_with_unit":   {"lang":"en","category":"argument_precision","user":"Tell me the weather in New York in Fahrenheit.","expected_tool":"get_weather","expected_args":["city","unit"]},
    "tool_selection_search":     {"lang":"en","category":"tool_selection","user":"Who won the 2024 UEFA Champions League final?","expected_tool":"web_search","expected_args":["query"]},
    "tool_selection_calculator": {"lang":"en","category":"tool_selection","user":"Calculate the square root of 144 multiplied by 7.","expected_tool":"calculate","expected_args":["expression"]},
    "tool_selection_convert":    {"lang":"en","category":"tool_selection","user":"I ran 7.2 km today. Convert that to miles.","expected_tool":"convert_units","expected_args":["kind","value"]},
    "no_tool_factual":     {"lang":"en","category":"no_tool","user":"What is the capital of France?","expected_tool":None,"expected_args":[]},
    "no_tool_greeting":    {"lang":"en","category":"no_tool","user":"Hello! How are you?","expected_tool":None,"expected_args":[]},
    "chain_search_then_email":  {"lang":"en","category":"multi_step","user":"Find out who won the latest Formula 1 race, then email the result to fan@racing.com with a suitable subject and body.","expected_tool":"web_search","expected_args":["query"],"second_expected_tool":"send_email","tool_response_1":'{"results": [{"snippet": "Max Verstappen won the 2025 Monaco Grand Prix."}]}'},
    "chain_calc_then_calendar": {"lang":"en","category":"multi_step","user":"I need a meeting that lasts 3 times 45 minutes. Calculate the total duration, then create a calendar event called 'Long Review' on 2025-08-20 at 09:00 with that duration.","expected_tool":"calculate","expected_args":["expression"],"second_expected_tool":"create_event","tool_response_1":'{"result": 135}'},
    "convert_temp":        {"lang":"en","category":"single_call","user":"Convert 37 degrees Celsius to Fahrenheit.","expected_tool":"convert_units","expected_args":["kind","value"]},
    "weather_implicit_city":{"lang":"en","category":"single_call","user":"Will I need an umbrella in Tokyo tomorrow?","expected_tool":"get_weather","expected_args":["city"]},
    "stock_with_currency": {"lang":"en","category":"argument_precision","user":"What is Tesla's stock price in EUR?","expected_tool":"get_stock_price","expected_args":["ticker","currency"]},
    "no_tool_definition":  {"lang":"en","category":"no_tool","user":"What does the word 'ephemeral' mean?","expected_tool":None,"expected_args":[]},
    "tool_select_stock_not_search":{"lang":"en","category":"tool_selection","user":"Look up the current price of Microsoft shares.","expected_tool":"get_stock_price","expected_args":["ticker"]},

    # ═══════════════ ITALIAN ═══════════════
    "meteo_base":          {"lang":"it","category":"single_call","user":"Che tempo fa a Roma in questo momento?","expected_tool":"get_weather","expected_args":["city"]},
    "calcolo_base":        {"lang":"it","category":"single_call","user":"Quanto fa (347 + 58) * 12?","expected_tool":"calculate","expected_args":["expression"]},
    "prezzo_azioni":       {"lang":"it","category":"single_call","user":"Qual è il prezzo attuale delle azioni Apple?","expected_tool":"get_stock_price","expected_args":["ticker"]},
    "invia_email_full":    {"lang":"it","category":"argument_precision","user":"Invia una email a marco@example.com con oggetto 'Riunione domani' e corpo 'Ciao Marco, promemoria per la riunione delle 10. Saluti, Luca.'","expected_tool":"send_email","expected_args":["to","subject","body"]},
    "crea_evento_full":    {"lang":"it","category":"argument_precision","user":"Crea un evento in calendario di 90 minuti intitolato 'Revisione Progetto' il 2025-07-10 alle 14:30.","expected_tool":"create_event","expected_args":["title","date","time","duration_minutes"]},
    "meteo_unita":         {"lang":"it","category":"argument_precision","user":"Dimmi il meteo a Milano in Fahrenheit.","expected_tool":"get_weather","expected_args":["city","unit"]},
    "sel_ricerca":         {"lang":"it","category":"tool_selection","user":"Chi ha vinto la finale di Champions League 2024?","expected_tool":"web_search","expected_args":["query"]},
    "sel_calcolo":         {"lang":"it","category":"tool_selection","user":"Calcola la radice quadrata di 144 moltiplicata per 7.","expected_tool":"calculate","expected_args":["expression"]},
    "sel_conversione":     {"lang":"it","category":"tool_selection","user":"Ho corso 7,2 km oggi. Convertili in miglia.","expected_tool":"convert_units","expected_args":["kind","value"]},
    "no_tool_capitale":    {"lang":"it","category":"no_tool","user":"Qual è la capitale della Francia?","expected_tool":None,"expected_args":[]},
    "no_tool_saluto":      {"lang":"it","category":"no_tool","user":"Ciao! Come stai?","expected_tool":None,"expected_args":[]},
    "no_tool_definizione": {"lang":"it","category":"no_tool","user":"Cosa significa la parola 'effimero'?","expected_tool":None,"expected_args":[]},
    "converti_temp":       {"lang":"it","category":"single_call","user":"Converti 37 gradi Celsius in Fahrenheit.","expected_tool":"convert_units","expected_args":["kind","value"]},
    "azioni_valuta":       {"lang":"it","category":"argument_precision","user":"Qual è il prezzo delle azioni Tesla in EUR?","expected_tool":"get_stock_price","expected_args":["ticker","currency"]},
    "catena_cerca_email":  {"lang":"it","category":"multi_step","user":"Scopri chi ha vinto l'ultima gara di Formula 1, poi invia il risultato via email a fan@racing.com con un oggetto e un corpo adeguati.","expected_tool":"web_search","expected_args":["query"],"second_expected_tool":"send_email","tool_response_1":'{"results": [{"snippet": "Max Verstappen ha vinto il Gran Premio di Monaco 2025."}]}'},
}

from collections import Counter
_lc = Counter(s["lang"] for s in TEST_SCENARIOS.values())
print(f"Defined {len(TEST_SCENARIOS)} scenarios  (EN={_lc['en']}, IT={_lc['it']}):")
for name, s in TEST_SCENARIOS.items():
    print(f"  [{s['lang']}] [{s['category']:18s}] {name}")

Defined 33 scenarios  (EN=18, IT=15):
  [en] [single_call       ] weather_basic
  [en] [single_call       ] calculator_basic
  [en] [single_call       ] stock_price
  [en] [argument_precision] send_email_full
  [en] [argument_precision] calendar_full_args
  [en] [argument_precision] weather_with_unit
  [en] [tool_selection    ] tool_selection_search
  [en] [tool_selection    ] tool_selection_calculator
  [en] [tool_selection    ] tool_selection_convert
  [en] [no_tool           ] no_tool_factual
  [en] [no_tool           ] no_tool_greeting
  [en] [multi_step        ] chain_search_then_email
  [en] [multi_step        ] chain_calc_then_calendar
  [en] [single_call       ] convert_temp
  [en] [single_call       ] weather_implicit_city
  [en] [argument_precision] stock_with_currency
  [en] [no_tool           ] no_tool_definition
  [en] [tool_selection    ] tool_select_stock_not_search
  [it] [single_call       ] meteo_base
  [it] [single_call       ] calcolo_base
  [it] [single_call       

## 📏 Scoring Helper

In [9]:
def score(answer, scenario):
    """
    Score a step-1 response against scenario expectations.
    Returns dict of boolean flags.
    """
    expected_tool = scenario["expected_tool"]
    expected_args = scenario.get("expected_args", [])
    tc = parse_tool_call(answer)

    s = {
        "tag_used":         bool(re.search(r"<tool_call>", answer)),
        "valid_json":       tc is not None,
        "correct_tool":     False,
        "all_args_present": False,
    }

    if expected_tool is None:          # no_tool: pass = no call produced
        no_call = tc is None
        s.update(valid_json=no_call, correct_tool=no_call, all_args_present=no_call)
        return s

    if tc is None:
        return s

    s["correct_tool"]     = (tc["name"] == expected_tool)
    s["all_args_present"] = all(a in tc["arguments"] for a in expected_args)
    return s


def ok(v):
    return "OK" if v else "--"


print("score() defined.")

score() defined.


## 🧪 Parser Self-Test
Verifies the parser on known output patterns before running the full benchmark.

In [10]:
test_cases = [
    ("native template format",
     '<tool_call>\n{"name": "get_weather", "arguments": {"city": "London"}}\n</tool_call>', "get_weather", "city"),
    ("compact format, no space",
     '<tool_call>\n{"name":"get_weather","arguments":{"city":"Roma"}}\n</tool_call>', "get_weather", "city"),
    ("single-quote Python repr (nesso style)",
     "{'name': 'calculate', 'arguments': {'expression': '3*45'}}", "calculate", "expression"),
    ("'tool' key variant, no tag",
     '{"tool": "send_email", "arguments": {"to": "a@b.com", "subject": "Hi", "body": "Hello"}}', "send_email", "to"),
    ("'tool_name' key variant",
     '{"tool_name": "get_stock_price", "arguments": {"ticker": "AAPL"}}', "get_stock_price", "ticker"),
    ("unclosed <tool_call> (stopped at eot)",
     '<tool_call>\n{"name": "calculate", "arguments": {"expression": "(1+2)*3"}}', "calculate", "expression"),
    ("string-encoded arguments",
     '<tool_call>\n{"name": "get_weather", "arguments": "{\\"city\\": \\"Rome\\"}"}\n</tool_call>', "get_weather", "city"),
    ("nested braces in a string value",
     '<tool_call>\n{"name": "send_email", "arguments": {"to": "a@b.c", "subject": "s", "body": "use {x} here"}}\n</tool_call>', "send_email", "body"),
    ("no tool call — plain text",
     "The capital of France is Paris.", None, None),
    ("plain text with unrelated braces",
     "In set notation {1, 2, 3} is a set.", None, None),
]
all_ok = True
for desc, text, exp_name, exp_arg in test_cases:
    tc = parse_tool_call(text)
    got = tc["name"] if tc else None
    has_arg = (exp_arg in tc["arguments"]) if (tc and exp_arg) else (exp_arg is None)
    passed = (got == exp_name) and has_arg
    all_ok &= passed
    print(f"[{'OK' if passed else 'FAIL'}] {desc}")
    if not passed: print(f"       expected={exp_name!r}  got={got!r}  parsed={tc}")

par = ('<tool_call>\n{"name":"get_weather","arguments":{"city":"Roma"}}\n</tool_call>\n'
       '<tool_call>\n{"name":"get_stock_price","arguments":{"ticker":"AAPL"}}\n</tool_call>')
names = [c["name"] for c in parse_tool_calls(par)]
ok_par = names == ["get_weather", "get_stock_price"]
all_ok &= ok_par
print(f"[{'OK' if ok_par else 'FAIL'}] parallel calls -> {names}")

# prompt-contract sanity: compact injection for a template without tools
_msgs = inject_tools([{"role": "system", "content": "S"}, {"role": "user", "content": "U"}], TOOLS[:1])
ok_inj = _msgs[0]["content"].startswith("S\n<tools>\n[") and _msgs[0]["content"].endswith("\n</tools>") or True
print()
print("All parser tests passed." if all_ok else "Some tests FAILED — fix parser before running benchmark.")

[OK] native template format
[OK] compact format, no space
[OK] single-quote Python repr (nesso style)
[OK] 'tool' key variant, no tag
[OK] 'tool_name' key variant
[OK] unclosed <tool_call> (stopped at eot)
[OK] string-encoded arguments
[OK] nested braces in a string value
[OK] no tool call — plain text
[OK] plain text with unrelated braces
[OK] parallel calls -> ['get_weather', 'get_stock_price']

All parser tests passed.


## 🚀 Run Benchmark

For each model × scenario:
- **Non-multi-step**: runs the full `tool_turn()` agentic loop
- **multi_step**: step 1 → inject pre-defined `tool_response_1` → step 2  
  (mirrors the `chain_*` pattern from the reference notebook)

In [11]:
results = {}

for model_id in MODELS:
    short = model_id.split("/")[-1]
    print(f"\n{'='*64}")
    print(f"  {short}")
    print(f"{'='*64}")

    try:
        tokenizer, model = load_model(model_id)
    except Exception as e:
        print(f"  ✗ Failed to load: {e}")
        results[model_id] = {}
        continue

    results[model_id] = {}
    print(f"  {'scenario':<28} {'cat':12} tag  json tool args  time  called")
    print(f"  {'-'*80}")

    for scenario_name, scenario in TEST_SCENARIOS.items():
        category  = scenario["category"]
        user_text = scenario["user"]

        try:
            t0 = time.time()

            if category == "multi_step":
                # ── Step 1 ──────────────────────────────────────────
                messages = [
                    {"role": "system", "content": system_for(scenario["lang"])},
                    {"role": "user",   "content": user_text},
                ]
                answer1 = chat(tokenizer, model, messages, tools=TOOLS)
                score1  = score(answer1, scenario)
                tc1     = parse_tool_call(answer1)

                # ── Inject tool_response_1 → Step 2 ─────────────────
                step2_answer = step2_score = None
                if "tool_response_1" in scenario:
                    messages2 = messages + [
                        {"role": "assistant", "content": answer1},
                        {"role": "tool",      "content": scenario["tool_response_1"]},
                    ]
                    step2_answer = chat(tokenizer, model, messages2, tools=TOOLS)
                    step2_score  = score(
                        step2_answer,
                        {"expected_tool": scenario["second_expected_tool"],
                         "expected_args": []}
                    )
                final_answer = step2_answer

            else:
                # ── Full tool_turn() agentic loop ────────────────────
                answer1, tc1, final_answer = tool_turn(tokenizer, model, user_text, lang=scenario["lang"])
                score1       = score(answer1, scenario)
                step2_answer = step2_score = None

            elapsed = time.time() - t0

            results[model_id][scenario_name] = {
                "answer":       answer1,
                "final_answer": final_answer,
                "score":        score1,
                "step2_answer": step2_answer,
                "step2_score":  step2_score,
                "time":         elapsed,
            }

            sc        = score1
            called    = (tc1["name"] if tc1 else "—") if category != "multi_step" else (
                (parse_tool_call(answer1) or {}).get("name", "—")
            )
            # flag mismatches so they stand out
            tool_flag = ok(sc.get('correct_tool'))
            args_flag = ok(sc.get('all_args_present'))
            marker    = "" if (sc.get('correct_tool') and sc.get('all_args_present')) else " ◄"
            print(
                f"  {scenario_name:<28} {category:12} "
                f"{ok(sc.get('tag_used')):4} "
                f"{ok(sc.get('valid_json')):4} "
                f"{tool_flag:4} "
                f"{args_flag:4} "
                f"{elapsed:4.1f}s "
                f"{called}{marker}"
            )

        except Exception as e:
            print(f"  {scenario_name:<28} ERROR: {e}")
            import traceback; traceback.print_exc()
            results[model_id][scenario_name] = {
                "answer": f"ERROR: {e}", "final_answer": None, "score": {},
                "step2_answer": None, "step2_score": None, "time": 0,
            }

    unload_model(model)
    print(f"  {short} unloaded.")

print("\n" + "="*64)
print("All benchmarks complete.")


  zagreus-0.4B-cpt-full-32k-sft-compact-v5b
  loading tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


  loading weights...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  67%|██████▋   | 194/290 [00:00<00:00, 1935.51it/s]

...

  no_tool_saluto               no_tool      --   OK   OK   OK    0.3s —


  no_tool_definizione          no_tool      --   OK   OK   OK    1.4s —


  converti_temp                single_call  OK   OK   OK   OK    1.3s convert_units


  azioni_valuta                argument_precision OK   OK   OK   OK    1.3s get_stock_price


  catena_cerca_email           multi_step   --   --   --   --    3.7s — ◄
  Qwen3-0.6B unloaded.

All benchmarks complete.


## 📊 Results — Per Scenario

In [12]:
for scenario_name, scenario in TEST_SCENARIOS.items():
    category = scenario["category"]
    user_msg = scenario["user"]
    short_q  = user_msg[:80] + ("..." if len(user_msg) > 80 else "")

    md_str  = f"### `{scenario_name}` [{category}]\n"
    md_str += f'*"{short_q}"*\n\n'
    md_str += "| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |\n"
    md_str += "|---|:---:|:---:|:---:|:---:|---:|---|\n"

    for model_id in MODELS:
        r     = results.get(model_id, {}).get(scenario_name)
        mname = model_id.split("/")[-1]
        if r is None:
            md_str += f"| `{mname}` | – | – | – | – | – | *not run* |\n"
            continue

        sc  = r.get("score", {})
        raw = r["answer"].replace("|", "\\|").replace("\n", " ")[:200]

        if scenario["expected_tool"] is None:
            no_call = parse_tool_call(r["answer"]) is None
            md_str += (
                f"| `{mname}` "
                f"| {'none' if not re.search(r'<tool_call>', r['answer']) else 'USED'} "
                f"| – "
                f"| {'OK' if no_call else 'SPURIOUS'} "
                f"| – "
                f"| {r['time']:.1f}s "
                f"| {raw} |\n"
            )
        else:
            md_str += (
                f"| `{mname}` "
                f"| {ok(sc.get('tag_used'))} "
                f"| {ok(sc.get('valid_json'))} "
                f"| {ok(sc.get('correct_tool'))} "
                f"| {ok(sc.get('all_args_present'))} "
                f"| {r['time']:.1f}s "
                f"| {raw} |\n"
            )

    # Multi-step: add step-2 table
    if category == "multi_step":
        s2tool = scenario.get("second_expected_tool", "?")
        md_str += f"\n**Step 2 (expected: `{s2tool}`):**\n\n"
        md_str += "| Model | Tag | JSON | Tool | Step-2 output |\n"
        md_str += "|---|:---:|:---:|:---:|---|\n"
        for model_id in MODELS:
            r     = results.get(model_id, {}).get(scenario_name)
            mname = model_id.split("/")[-1]
            if r is None or r.get("step2_score") is None:
                md_str += f"| `{mname}` | – | – | – | – |\n"
                continue
            sc2  = r["step2_score"]
            raw2 = (r["step2_answer"] or "").replace("|", "\\|").replace("\n", " ")[:200]
            md_str += (
                f"| `{mname}` "
                f"| {ok(sc2.get('tag_used'))} "
                f"| {ok(sc2.get('valid_json'))} "
                f"| {ok(sc2.get('correct_tool'))} "
                f"| {raw2} |\n"
            )

    display(Markdown(md_str + "\n---\n"))

### `weather_basic` [single_call]
*"What is the weather like in London right now?"*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | OK | OK | 1.3s | <tool_call> {"name": "get_weather", "arguments": {"city": "London"}} </tool_call> |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | OK | 1.0s | <tool_call> {"name": "get_weather", "arguments": {"city": "London"}} </tool_call> |
| `Qwen3-0.6B` | OK | OK | OK | OK | 1.1s | <tool_call> {"name": "get_weather", "arguments": {"city": "London"}} </tool_call> |

---


### `calculator_basic` [single_call]
*"What is (347 + 58) * 12?"*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | -- | -- | -- | -- | 0.6s | I'm sorry, I'm unable to perform that calculation as I don't have the capability to perform mathematical operations. |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | -- | -- | -- | -- | 0.8s | I'm sorry, I can't perform that calculation. I'm a text-based AI, and I don't have the ability to perform mathematical operations. |
| `Qwen3-0.6B` | OK | OK | OK | OK | 1.2s | <tool_call> {"name": "calculate", "arguments": {"expression": "(347 + 58) * 12"}} </tool_call> |

---


### `stock_price` [single_call]
*"What is Apple's current stock price?"*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | OK | OK | 1.0s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "AAPL"}} </tool_call> |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | OK | 1.2s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "AAPL"}} </tool_call> |
| `Qwen3-0.6B` | OK | OK | OK | OK | 1.2s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "AAPL", "currency": "USD"}} </tool_call> |

---


### `send_email_full` [argument_precision]
*"Send an email to alice@example.com with subject 'Meeting tomorrow' and body 'Hi ..."*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | OK | OK | 2.7s | <tool_call> {"name": "send_email", "arguments": {"to": "alice@example.com", "subject": "Meeting tomorrow", "body": "Hi Alice, reminder about our 10am meeting. Best, Bob."}} </tool_call> |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | OK | 1.6s | <tool_call> {"name": "send_email", "arguments": {"to": "alice@example.com", "subject": "Meeting Tomorrow", "body": "Hi Alice, reminder about our 10am meeting. Best, Bob."}} </tool_call> |
| `Qwen3-0.6B` | OK | OK | OK | OK | 2.4s | <tool_call> {"name": "send_email", "arguments": {"to": "alice@example.com", "subject": "Meeting tomorrow", "body": "Hi Alice, reminder about our 10am meeting. Best, Bob."}} </tool_call> |

---


### `calendar_full_args` [argument_precision]
*"Create a 90-minute calendar event called 'Project Review' on 2025-07-10 at 14:30..."*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | OK | -- | 1.8s | <tool_call> {"name": "create_event", "arguments": {"title": "Project Review", "date": "2025-07-10", "time": "14:30"}} </tool_call> |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | -- | 1.6s | <tool_call> {"name": "create_event", "arguments": {"title": "Project Review", "date": "2025-07-10", "time": "14:30"}} </tool_call> |
| `Qwen3-0.6B` | OK | OK | OK | OK | 2.9s | <tool_call> {"name": "create_event", "arguments": {"title": "Project Review", "date": "2025-07-10", "time": "14:30", "duration_minutes": 90}} </tool_call> |

---


...

### `no_tool_saluto` [no_tool]
*"Ciao! Come stai?"*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | none | – | OK | – | 0.4s | Ciao! Sto bene, grazie. Come posso aiutarti oggi? |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | none | – | OK | – | 0.2s | Ciao! Come stai? |
| `Qwen3-0.6B` | none | – | OK | – | 0.3s | Ciao! Sto bene. Come stai? |

---


### `no_tool_definizione` [no_tool]
*"Cosa significa la parola 'effimero'?"*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | none | – | OK | – | 0.6s | Un effimero è un oggetto o un'entità che non ha una durata definita. |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | none | – | OK | – | 0.5s | La parola "effimero" significa "che non ha un significato duraturo". |
| `Qwen3-0.6B` | none | – | OK | – | 1.4s | La parola "effimero" significa "effemero" o "effemero" in inglese, che è un termine usato per indicare un'esperienza o un'esperienza di vita breve e breve. |

---


### `converti_temp` [single_call]
*"Converti 37 gradi Celsius in Fahrenheit."*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | OK | OK | 1.5s | <tool_call> {"name": "convert_units", "arguments": {"kind": "c", "value": 37}} </tool_call> |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | OK | 1.7s | <tool_call> {"name": "convert_units", "arguments": {"kind": "celsius", "value": 37}} </tool_call> |
| `Qwen3-0.6B` | OK | OK | OK | OK | 1.3s | <tool_call> {"name": "convert_units", "arguments": {"kind": "c_to_f", "value": 37}} </tool_call> |

---


### `azioni_valuta` [argument_precision]
*"Qual è il prezzo delle azioni Tesla in EUR?"*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | OK | -- | 1.2s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "TSLA"}} </tool_call> |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | -- | 1.2s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "TSLA"}} </tool_call> |
| `Qwen3-0.6B` | OK | OK | OK | OK | 1.3s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "TSLA", "currency": "EUR"}} </tool_call> |

---


### `catena_cerca_email` [multi_step]
*"Scopri chi ha vinto l'ultima gara di Formula 1, poi invia il risultato via email..."*

| Model | Tag | JSON | Tool | Args | Time | Step-1 output (truncated) |
|---|:---:|:---:|:---:|:---:|---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | -- | -- | 4.5s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "F1"}} </tool_call> <tool_call> {"name": "send_email", "arguments": {"to": "fan@racing.com", "subject": "Risultato gara Formula 1", "bod |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | -- | -- | 2.5s | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "F1"}} </tool_call> |
| `Qwen3-0.6B` | -- | -- | -- | -- | 3.7s | 1. **Scopri chi ha vinto l'ultima gara di Formula 1**:    - Utilizza la funzione `web_search` per cercare informazioni su questa ultima gara.  2. **Invia il risultato via email a fan@racing.com**:     |

**Step 2 (expected: `send_email`):**

| Model | Tag | JSON | Tool | Step-2 output |
|---|:---:|:---:|:---:|---|
| `zagreus-0.4B-cpt-full-32k-sft-compact-v5b` | OK | OK | -- | <tool_call> {"name": "get_stock_price", "arguments": {"ticker": "F1"}} </tool_call> <tool_call> {"name": "send_email", "arguments": {"to": "fan@racing.com", "subject": "Risultato gara Formula 1", "bod |
| `zagreus-0.4B-cpt-full-32k-sft-compact-v6` | OK | OK | OK | <tool_call> {"name": "send_email", "arguments": {"to": "fan@racing.com", "subject": "Gara di Formula 1 2025: chi ha vinto?", "body": "Gara di Formula 1 2025: chi ha vinto il Gran Premio di Monaco."}}  |
| `Qwen3-0.6B` | -- | -- | -- | Max Verstappen ha vinto l'ultima gara di Formula 1. Invia il risultato via email a fan@racing.com con un oggetto e un corpo adeguati. |

---


## 🏆 Aggregate Scoreboard

In [13]:
categories = ["single_call", "argument_precision", "tool_selection", "no_tool", "multi_step"]

def passed_scenario(model_id, sname, scenario):
    r = results.get(model_id, {}).get(sname)
    if r is None: return None
    sc = r.get("score", {})
    if scenario["expected_tool"] is None:
        return parse_tool_call(r["answer"]) is None
    return sc.get("correct_tool", False) and sc.get("all_args_present", False)

scoreboard = {}
lang_board = {}
for model_id in MODELS:
    mname = model_id.split("/")[-1]
    scoreboard[mname] = {c: [] for c in categories}; scoreboard[mname]["overall"] = []
    lang_board[mname] = {"en": [], "it": []}
    for sname, scenario in TEST_SCENARIOS.items():
        p = passed_scenario(model_id, sname, scenario)
        if p is None: continue
        cat = scenario["category"]
        if cat in scoreboard[mname]: scoreboard[mname][cat].append(p)
        scoreboard[mname]["overall"].append(p)
        lang_board[mname][scenario["lang"]].append(p)

def pct(lst): return f"{100*sum(lst)//len(lst)}%" if lst else "–"

W = 18
header = f"{'Model':<32}"
for c in categories: header += f" {c[:W]:>{W}}"
header += f" {'OVERALL':>{W}}"
print(header); print("-" * (32 + (W+1) * (len(categories)+1)))
for mname, data in scoreboard.items():
    row = f"{mname:<32}"
    for c in categories: row += f" {pct(data[c]):>{W}}"
    row += f" {pct(data['overall']):>{W}}"
    print(row)

print("\n── By language ──")
print(f"{'Model':<32} {'IT':>8} {'EN':>8} {'OVERALL':>10}")
print("-" * 60)
for mname in scoreboard:
    lb = lang_board[mname]
    print(f"{mname:<32} {pct(lb['it']):>8} {pct(lb['en']):>8} {pct(scoreboard[mname]['overall']):>10}")

Model                                   single_call argument_precision     tool_selection            no_tool         multi_step            OVERALL
--------------------------------------------------------------------------------------------------------------------------------------------------
zagreus-0.4B-cpt-full-32k-sft-compact-v5b                77%                25%                14%               100%                 0%                48%
zagreus-0.4B-cpt-full-32k-sft-compact-v6                66%                25%                 0%               100%                33%                45%
Qwen3-0.6B                                      77%               100%                71%               100%                66%                84%

── By language ──
Model                                  IT       EN    OVERALL
------------------------------------------------------------
zagreus-0.4B-cpt-full-32k-sft-compact-v5b      46%      50%        48%
zagreus-0.4B-cpt-full-32k-sft-compa

## ⏱️ Timing Summary

In [14]:
print(f"{'Model':<35} {'Avg (s)':>10} {'Total (s)':>12}")
print("-" * 60)
for model_id in MODELS:
    mname = model_id.split("/")[-1]
    times = [v["time"] for v in results.get(model_id, {}).values() if v.get("time", 0) > 0]
    if times:
        print(f"{mname:<35} {sum(times)/len(times):>10.2f} {sum(times):>12.2f}")
    else:
        print(f"{mname:<35} {'N/A':>10} {'N/A':>12}")

Model                                  Avg (s)    Total (s)
------------------------------------------------------------
zagreus-0.4B-cpt-full-32k-sft-compact-v5b       1.24        40.88
zagreus-0.4B-cpt-full-32k-sft-compact-v6       1.03        34.08
Qwen3-0.6B                                1.43        47.07


## 🔍 Raw Output Inspector
Set the two variables below to inspect any model × scenario in full.

In [15]:
INSPECT_MODEL    = "mii-llm/nesso-0.4B-agentic"  # ← change me
INSPECT_SCENARIO = "chain_search_then_email"        # ← change me

r = results.get(INSPECT_MODEL, {}).get(INSPECT_SCENARIO)
if r:
    sc = r.get("score", {})
    print(f"Model    : {INSPECT_MODEL}")
    print(f"Scenario : {INSPECT_SCENARIO}")
    print(f"Time     : {r['time']:.2f}s")
    print(f"Scores   : tag={sc.get('tag_used')}  json={sc.get('valid_json')}  "
          f"tool={sc.get('correct_tool')}  args={sc.get('all_args_present')}")
    print("\n--- Step 1 answer ---")
    print(r["answer"])
    print("\n--- Parsed tool call ---")
    tc = parse_tool_call(r["answer"])
    print(json.dumps(tc, indent=2) if tc else "(none)")
    if r.get("final_answer") and r["final_answer"] != r["answer"]:
        print("\n--- Final answer ---")
        print(r["final_answer"])
    if r.get("step2_answer"):
        print("\n--- Step 2 answer ---")
        print(r["step2_answer"])
else:
    print("No result found. Check INSPECT_MODEL and INSPECT_SCENARIO.")

No result found. Check INSPECT_MODEL and INSPECT_SCENARIO.
